In [1]:
#core library
import numpy as np
import pandas as pd

In [2]:
#ML utilities
from sklearn.model_selection import(
train_test_split,StratifiedKFold,
RandomizedSearchCV)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import(
accuracy_score,
precision_score,
recall_score,
f1_score,
roc_auc_score,
classification_report,
confusion_matrix)

from sklearn.inspection import permutation_importance


In [3]:
np.random.seed(42)

n_samples = 1000

df = pd.DataFrame({
    "Age": np.random.randint(21, 65, n_samples),
    "Income": np.random.randint(20000, 150000, n_samples),
    "Credit_Score": np.random.randint(300, 850, n_samples),
    "Loan_Amount": np.random.randint(50000, 800000, n_samples),
    "Employment_Years": np.random.randint(0, 30, n_samples),
    "Loan_Approved": np.random.choice([0, 1], size=n_samples, p=[0.45, 0.55])
})

In [4]:
df.head()

,Age,Income,Credit_Score,Loan_Amount,Employment_Years,Loan_Approved
0,59,24018,505,187888,20,1
1,49,31302,754,127175,21,1
2,35,87506,572,638595,20,1
3,63,61157,535,358877,7,1
4,28,74917,820,145028,16,0


In [5]:
#feature and target seperation
X=df.drop("Loan_Approved", axis=1)
y=df["Loan_Approved"]

In [6]:
X_train, X_test,y_train,y_test=train_test_split(X,y,
                                                test_size=0.25,
                                                stratify=y,
                                                random_state=42
                                               )

In [7]:
numeric_features = ['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Employment_Years']
# List of all numeric columns in the dataset

In [8]:
X.columns

Index(['Age', 'Income', 'Credit_Score', 'Loan_Amount', 'Employment_Years'], dtype='object')

In [9]:
numeric_pipeline=Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

In [10]:
preprocessor = ColumnTransformer(transformers=[
    ("num",numeric_pipeline,numeric_features)])

In [11]:
rf_model = RandomForestClassifier(
    n_estimators=308,        # number of trees in the forest
    max_depth=None,          # no limit → trees grow fully
    min_samples_split=5,     # minimum samples required to split a node
    min_samples_leaf=2,      # minimum samples required at leaf node
    max_features="sqrt",     # number of features to consider at each split
    class_weight="balanced", # handles imbalanced dataset
    oob_score=True,          # use out-of-bag samples for validation
    n_jobs=-1,               # use all CPU cores for faster training
    random_state=42          # for reproducibility
)

In [12]:
# n_estimators=300 → Uses 300 decision trees to improve prediction stability
# max_depth=None → Trees grow fully without depth restriction
# min_samples_split=5 → At least 5 samples are needed to split a node
# min_samples_leaf=2 → Each leaf must have at least 2 samples
# max_features="sqrt" → Uses √(features) at each split for randomness
# class_weight="balanced" → Automatically adjusts weights for imbalanced classes
# oob_score=True → Uses out-of-bag samples to estimate model accuracy
# n_jobs=-1 → Uses all CPU cores for faster computation
# random_state=42 → Ensures reproducible results

In [13]:
model_pipeline=Pipeline(steps=[
    ("preprocessing",preprocessor),
    ("model", rf_model)
])  #full pipeline

In [14]:
#model trining
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Income',
                                                   'Credit_Score',
                                                   'Loan_Amount',
                                                   'Employment_Years'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced',
                                        min_samples_leaf=2, min_samples_split=5,
                                        n_estimators=308, n_jobs=-1,
                                        oob_score=True, random_state=42))])

In [15]:
# OOB Score
print("OOB Score:", model_pipeline.named_steps["model"].oob_score_)

OOB Score: 0.524


In [16]:
y_pred= model_pipeline.predict(X_test)
y_prob=model_pipeline.predict_proba(X_test)[:,1]

In [17]:
print("advanced evaluation matrics")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))

print("Recall :", recall_score(y_test, y_pred))

print("F1-Score", f1_score(y_test, y_pred))

print("ROC-AUC:", roc_auc_score (y_test, y_prob))

print("\nClassification Report")
print(classification_report(y_test,y_pred))

advanced evaluation matrics
Accuracy: 0.508
Precision: 0.5285714285714286
Recall : 0.5648854961832062
F1-Score 0.5461254612546126
ROC-AUC: 0.5204310731926358

Classification Report
              precision    recall  f1-score   support

           0       0.48      0.45      0.46       119
           1       0.53      0.56      0.55       131

    accuracy                           0.51       250
   macro avg       0.51      0.51      0.50       250
weighted avg       0.51      0.51      0.51       250



In [18]:
feature_names = numeric_features

importances = model_pipeline.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importance (Gini):")
print(feature_importance_df)


Feature Importance (Gini):
            Feature  Importance
1            Income    0.223409
3       Loan_Amount    0.222186
2      Credit_Score    0.214669
0               Age    0.181742
4  Employment_Years    0.157994


In [19]:
# Calculate permutation importance using test data
perm_importance = permutation_importance(
    model_pipeline,   # trained pipeline model
    X_test,           # test features
    y_test,           # test target
    n_repeats=10,     # number of shuffles
    random_state=42,
    n_jobs=-1         # use all CPU cores
)

# Create DataFrame for better visualization
perm_df = pd.DataFrame({
    "Feature": feature_names,
    "Permutation_Importance": perm_importance.importances_mean
}).sort_values(by="Permutation_Importance", ascending=False)

# Print results
print("\nPermutation Feature Importance")
print(perm_df)


Permutation Feature Importance
            Feature  Permutation_Importance
3       Loan_Amount                  0.0124
2      Credit_Score                 -0.0056
4  Employment_Years                 -0.0108
0               Age                 -0.0136
1            Income                 -0.0180


In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# Create Stratified K-Fold object (keeps class balance in each fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = []  # store F1 scores

# Loop through each fold
for train_idx, val_idx in skf.split(X, y):

    # Split data into training and validation sets
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train model on training data
    model_pipeline.fit(X_tr, y_tr)

    # Predict on validation data
    preds = model_pipeline.predict(X_val)

    # Calculate F1 score and store it
    cv_scores.append(f1_score(y_val, preds))

# Print all fold scores
print("\nStratified CV F1 Scores:", cv_scores)

# Print average score
print("Mean CV F1 Score:", np.mean(cv_scores))


Stratified CV F1 Scores: [0.5650224215246636, 0.5701754385964912, 0.5167464114832536, 0.6094420600858369, 0.5454545454545454]
Mean CV F1 Score: 0.5613681754289581


In [21]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter distribution for Random Forest inside pipeline
param_dist = {
    "model__n_estimators": [200, 300, 500],        # number of trees
    "model__max_depth": [None, 10, 20, 30],        # depth of trees
    "model__min_samples_split": [2, 5, 10],        # min samples to split
    "model__min_samples_leaf": [1, 2, 4],          # min samples at leaf
    "model__max_features": ["sqrt", "log2"]        # features per split
}

# Create RandomizedSearchCV object
random_search = RandomizedSearchCV(
    model_pipeline,             # your pipeline
    param_distributions=param_dist,
    n_iter=20,                  # number of random combinations
    scoring="f1",               # evaluation metric
    cv=5,                       # 5-fold cross validation
    n_jobs=-1,                  # use all CPU cores
    random_state=42
)
random_search.fit(X_train,y_train)

RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['Age',
                                                                                'Income',
                                                                                'Credit_Score',
                                                                                'Loan_Amount',
                                                                                'Employment_Years'])])),
                                             ('model',
                                              RandomForestClassifier(class_weight='balanced',
                                                                     min_samples_leaf=2,
                                                                     min_samples_split=5,
                                                                     n_estimators=308,
                                                                     n_jobs=-1,
                                                                     oob_score=True,
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [None, 10, 20, 30],
                                        'model__max_features': ['sqrt', 'log2'],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [200, 300, 500]},
                   random_state=42, scoring='f1')

In [24]:
#best model after tuning
best_model=random_search.best_estimator_
print("\n Best Hyperparameters")
print(random_search.best_params_)


 Best Hyperparameters
{'model__n_estimators': 200, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': 'sqrt', 'model__max_depth': 30}


In [25]:
#Final tuned model evaluation
final_pred=best_model.predict(X_test)
final_prob=best_model.predict_proba(X_test)[:,1]

print("\nFinal tuned performance")
print("Accuracy:", accuracy_score(y_test,final_pred))
print("precision:", precision_score(y_test,final_pred))
print("recall:", f1_score(y_test,final_pred))
print("ROC AUC:", roc_auc_score(y_test,final_pred))


Final tuned performance
Accuracy: 0.524
precision: 0.5461538461538461
recall: 0.5440613026819924
ROC AUC: 0.5230932067483482
